# Bank of Canada Policy Rate Decision Model

Same pipeline philosophy as the Lotto Max notebook, retargeted to a problem that actually
has signal: **predict the next Bank of Canada decision (cut / hold / hike)**.

What it does, in order:

1. Downloads everything systematically, no keys needed: BoC Valet (policy rate, T-bills,
   GoC 2y/10y/long yields, CPI core measures), FRED (unemployment, Fed funds, US 2y, WTI,
   USD/CAD), Yahoo Finance (ZFL, Canadian sector ETFs, US sector ETFs, TSX 60).
2. Builds the full decision calendar (fixed announcement dates since 2001, plus any
   unscheduled move detected in the rate series) and labels each meeting.
3. Engineers features **as of the day before each decision**, with publication lags on
   macro data so there is no look-ahead.
4. Tunes Logistic Regression, XGBoost, MLP and an LSTM (daily sequences) with a
   time-series CV, then runs a **walk-forward** test: refit before every meeting, predict
   it, move on.
5. Picks the winner on a validation window and reports it on a **separate holdout**
   window, against honest baselines (always-hold, persistence, market-only).
6. Predicts the next meeting with probabilities.

Design notes from the economist side:

- **ZFL** (long federal bonds) only exists since 2010. Before that its return is proxied
  as `-duration x change in GoC long yield`, then spliced with the real ETF.
- **BMO ZXL\* ETFs** are US sector SPDRs held in CAD. Their histories are short, so the
  notebook rebuilds them from the US ETF x USD/CAD (unhedged equivalent), which goes back
  20+ years. Canadian-native sector ETFs are used as-is; late starters stay NaN early.
- The single most informative input is usually the market itself (3-month T-bill and
  2y yield vs the policy rate). The **market-only** baseline is the bar to beat; if the
  full model can't beat it on the holdout, the extra features aren't adding information.
- ~200 decisions in total. That is a small sample, so models are kept regularized and
  results should be read as probabilities, not certainties.

## 1. Setup

In [ ]:
!pip install xgboost yfinance -q

In [ ]:
import io, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import yfinance as yf
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import log_loss, accuracy_score, f1_score, confusion_matrix
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
np.random.seed(42)
pd.set_option("display.max_columns", 80, "display.width", 160)

START       = "2001-01-01"   # data start (fixed BoC announcement dates began Dec 2000)
FIRST_MEET  = "2002-01-01"   # first meeting used as a sample (needs ~1y of history)
VALID_START = "2012-01-01"   # walk-forward begins; 2012-2018 = validation (model selection)
TEST_START  = "2019-01-01"   # 2019+ = untouched holdout (reported score)
CLASSES     = ["cut", "hold", "hike"]          # encoded 0, 1, 2
LABEL_EPS   = 0.05                              # min move (pp) to count as cut/hike
ZFL_DURATION = 15.0                             # approx modified duration for the ZFL proxy
HEADERS = {"User-Agent": "Mozilla/5.0 (research notebook)"}

## 2. Decision calendar

Fixed announcement dates published by the BoC. Small date errors here are harmless: every
actual rate change in the policy-rate series is detected automatically and snapped to the
nearest scheduled date (or added as an unscheduled meeting, e.g. March 2020). Add each new
year's dates to keep the calendar current; 2026 is already in.

In [ ]:
FAD = {
 2001: ["01-23","03-06","04-17","05-29","07-17","08-28","10-23","11-27"],
 2002: ["01-15","03-05","04-16","06-04","07-16","09-04","10-16","12-03"],
 2003: ["01-21","03-04","04-15","06-03","07-15","09-03","10-15","12-02"],
 2004: ["01-20","03-02","04-13","06-08","07-20","09-08","10-19","12-07"],
 2005: ["01-25","03-01","04-12","05-25","07-12","09-07","10-18","12-06"],
 2006: ["01-24","03-07","04-25","05-24","07-11","09-06","10-17","12-05"],
 2007: ["01-16","03-06","04-24","06-05","07-10","09-05","10-16","12-04"],
 2008: ["01-22","03-04","04-22","06-10","07-15","09-03","10-21","12-09"],
 2009: ["01-20","03-03","04-21","06-04","07-23","09-10","10-20","12-08"],
 2010: ["01-19","03-02","04-20","06-01","07-20","09-08","10-19","12-07"],
 2011: ["01-18","03-01","04-12","05-31","07-19","09-07","10-25","12-06"],
 2012: ["01-17","03-08","04-17","06-05","07-17","09-05","10-23","12-04"],
 2013: ["01-23","03-06","04-17","05-29","07-17","09-04","10-23","12-04"],
 2014: ["01-22","03-05","04-16","06-04","07-16","09-03","10-22","12-03"],
 2015: ["01-21","03-04","04-15","05-27","07-15","09-09","10-21","12-02"],
 2016: ["01-20","03-09","04-13","05-25","07-13","09-07","10-19","12-07"],
 2017: ["01-18","03-01","04-12","05-24","07-12","09-06","10-25","12-06"],
 2018: ["01-17","03-07","04-18","05-30","07-11","09-05","10-24","12-05"],
 2019: ["01-09","03-06","04-24","05-29","07-10","09-04","10-30","12-04"],
 2020: ["01-22","03-04","04-15","06-03","07-15","09-09","10-28","12-09"],
 2021: ["01-20","03-10","04-21","06-09","07-14","09-08","10-27","12-08"],
 2022: ["01-26","03-02","04-13","06-01","07-13","09-07","10-26","12-07"],
 2023: ["01-25","03-08","04-12","06-07","07-12","09-06","10-25","12-06"],
 2024: ["01-24","03-06","04-10","06-05","07-24","09-04","10-23","12-11"],
 2025: ["01-29","03-12","04-16","06-04","07-30","09-17","10-29","12-10"],
 2026: ["01-28","03-18","04-29","06-10","07-15","09-02","10-28","12-09"],
}
SCHEDULED = pd.DatetimeIndex(sorted(pd.Timestamp(f"{y}-{d}") for y, ds in FAD.items() for d in ds))
print(f"{len(SCHEDULED)} scheduled dates, {SCHEDULED[0].date()} .. {SCHEDULED[-1].date()}")

## 3. Download all data

Each series is tried from a list of sources/IDs and the first that works is used, so a
renamed series or a down site degrades gracefully instead of crashing the notebook. A
coverage report prints at the end.

In [ ]:
def valet(series_id, start=START):
    url = f"https://www.bankofcanada.ca/valet/observations/{series_id}/json?start_date={start}"
    r = requests.get(url, headers=HEADERS, timeout=30); r.raise_for_status()
    obs = r.json()["observations"]
    s = pd.Series({pd.Timestamp(o["d"]): o.get(series_id, {}).get("v") for o in obs})
    s = pd.to_numeric(s, errors="coerce").dropna().sort_index()
    if s.empty: raise ValueError("empty")
    return s

def fred(series_id, start=START):
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}&cosd={start}"
    r = requests.get(url, headers=HEADERS, timeout=30); r.raise_for_status()
    df = pd.read_csv(io.StringIO(r.text))
    s = pd.to_numeric(df.iloc[:, 1], errors="coerce")
    s.index = pd.to_datetime(df.iloc[:, 0]); s = s.dropna().sort_index()
    if s.empty: raise ValueError("empty")
    return s

def first_ok(name, attempts):
    # primary = first source that works; later sources only backfill dates before it starts
    out = None
    for src, sid in attempts:
        try:
            got = (valet if src == "valet" else fred)(sid)
        except Exception:
            continue
        if out is None:
            out = got; print(f"  {name:12s} <- {src}:{sid}  ({got.index[0].date()} .. {got.index[-1].date()})")
        else:
            early = got[got.index < out.index[0]]
            if len(early):
                out = pd.concat([early, out]).sort_index()
                print(f"  {'':12s}    backfilled from {src}:{sid} to {early.index[0].date()}")
    if out is None:
        print(f"  {name:12s} MISSING"); return pd.Series(dtype=float, name=name)
    return out.rename(name)

# BoC target overnight rate: every change since 2001 (date of announcement -> new rate).
# Used to backfill the policy rate if the Valet daily series starts late.
POLICY_CHANGES = {
 "2001-01-01":5.75,"2001-01-23":5.50,"2001-03-06":5.00,"2001-04-17":4.75,"2001-05-29":4.50,
 "2001-07-17":4.25,"2001-08-28":4.00,"2001-09-17":3.50,"2001-10-23":2.75,"2001-11-27":2.25,
 "2002-01-15":2.00,"2002-04-16":2.25,"2002-06-04":2.50,"2002-07-16":2.75,
 "2003-03-04":3.00,"2003-04-15":3.25,"2003-07-15":3.00,"2003-09-03":2.75,
 "2004-01-20":2.50,"2004-03-02":2.25,"2004-04-13":2.00,"2004-09-08":2.25,"2004-10-19":2.50,
 "2005-09-07":2.75,"2005-10-18":3.00,"2005-12-06":3.25,
 "2006-01-24":3.50,"2006-03-07":3.75,"2006-04-25":4.00,"2006-05-24":4.25,
 "2007-07-10":4.50,"2007-12-04":4.25,
 "2008-01-22":4.00,"2008-03-04":3.50,"2008-04-22":3.00,"2008-10-08":2.50,"2008-10-21":2.25,"2008-12-09":1.50,
 "2009-01-20":1.00,"2009-03-03":0.50,"2009-04-21":0.25,
 "2010-06-01":0.50,"2010-07-20":0.75,"2010-09-08":1.00,
 "2015-01-21":0.75,"2015-07-15":0.50,
 "2017-07-12":0.75,"2017-09-06":1.00,
 "2018-01-17":1.25,"2018-07-11":1.50,"2018-10-24":1.75,
 "2020-03-04":1.25,"2020-03-13":0.75,"2020-03-27":0.25,
 "2022-03-02":0.50,"2022-04-13":1.00,"2022-06-01":1.50,"2022-07-13":2.50,"2022-09-07":3.25,
 "2022-10-26":3.75,"2022-12-07":4.25,"2023-01-25":4.50,"2023-06-07":4.75,"2023-07-12":5.00,
 "2024-06-05":4.75,"2024-07-24":4.50,"2024-09-04":4.25,"2024-10-23":3.75,"2024-12-11":3.25,
 "2025-01-29":3.00,"2025-03-12":2.75,"2025-09-17":2.50,"2025-10-29":2.25,
}
_pc = pd.Series(POLICY_CHANGES); _pc.index = pd.to_datetime(_pc.index)
policy_hist = _pc.reindex(pd.bdate_range(START, _pc.index[-1]).union(_pc.index)).ffill()

print("Daily rates")
policy  = first_ok("policy",  [("valet","V39079")])
early = policy_hist[policy_hist.index < (policy.index[0] if len(policy) else pd.Timestamp.max)]
policy = pd.concat([early, policy]).sort_index().rename("policy")   # Valet wins where it exists
print(f"  policy rate final coverage: {policy.index[0].date()} .. {policy.index[-1].date()}")
tbill3m = first_ok("tbill3m", [("valet","V39065"), ("valet","TB.CDN.90D.MID"), ("fred","IR3TIB01CAM156N")])
y2      = first_ok("y2",      [("valet","BD.CDN.2YR.DQ.YLD"), ("valet","V39051")])
y10     = first_ok("y10",     [("valet","BD.CDN.10YR.DQ.YLD"), ("valet","V39055"), ("fred","IRLTLT01CAM156N")])
ylong   = first_ok("ylong",   [("valet","BD.CDN.LONG.DQ.YLD"), ("valet","V39056")])
fedf    = first_ok("fedfunds",[("fred","DFF")])
us2     = first_ok("us2y",    [("fred","DGS2")])
wti     = first_ok("wti",     [("fred","DCOILWTICO")])
usdcad  = first_ok("usdcad",  [("fred","DEXCAUS"), ("valet","FXUSDCAD")])

print("Monthly macro (Canada)")
cpi_tot = first_ok("cpi_yoy",   [("valet","STATIC_TOTALCPICHANGE"), ("fred","CPALTT01CAM659N")])
cpi_trm = first_ok("cpi_trim",  [("valet","CPI_TRIM")])
cpi_med = first_ok("cpi_median",[("valet","CPI_MEDIAN")])
unemp   = first_ok("unemp",     [("fred","LRUNTTTTCAM156S"), ("fred","LRHUTTTTCAM156S")])

In [ ]:
# ---- ETFs -------------------------------------------------------------------------
# Canadian-native sector ETFs (user list) + ZFL + TSX 60 benchmark
CA_ETF = {"ZFL":"ZFL.TO", "XIU":"XIU.TO",
          "XUT":"XUT.TO", "ZUT":"ZUT.TO", "ZUH":"ZUH.TO", "XCD":"XCD.TO", "ZMT":"ZMT.TO",
          "ZDJ":"ZDJ.TO", "XMA":"XMA.TO", "XFN":"XFN.TO", "ZEB":"ZEB.TO", "XEG":"XEG.TO",
          "XRE":"XRE.TO", "XIT":"XIT.TO", "XST":"XST.TO"}
# US sector SPDRs -> rebuilt in CAD = what BMO's ZXL* (unhedged) hold
US_ETF = {"ZXLU":"XLU", "ZXLV":"XLV", "ZXLY":"XLY", "ZXLC":"XLC", "ZXLI":"XLI", "ZXLB":"XLB",
          "ZXLF":"XLF", "ZXLE":"XLE", "ZXLR":"VNQ", "ZXLK":"XLK", "ZXLP":"XLP"}

def yahoo(tickers):
    out = {}
    for name, tk in tickers.items():
        for attempt in range(3):
            try:
                df = yf.download(tk, start=START, auto_adjust=True, progress=False, threads=False)
                if df is None or df.empty: raise ValueError("empty")
                px = df["Close"]; px = px.iloc[:, 0] if isinstance(px, pd.DataFrame) else px
                out[name] = px.dropna().rename(name); break
            except Exception:
                time.sleep(1.5)
        status = f"{out[name].index[0].date()} .. {out[name].index[-1].date()}" if name in out else "MISSING"
        print(f"  {name:5s} ({tk:7s}) {status}")
    return out

print("Canadian ETFs");  ca_px = yahoo(CA_ETF)
print("US sector ETFs"); us_px = yahoo(US_ETF)

## 4. Daily panel, splices, and publication lags

- Business-day panel, forward-filled (holidays differ between Canada and the US).
- **ZFL splice**: before ZFL existed, daily return = `-15 x change in GoC long yield / 100`.
- **ZXL\* rebuild**: US ETF price x USD/CAD.
- Macro series are stamped with the date they would actually have been public
  (CPI about 3 weeks after month end, LFS about 10 days), then forward-filled. This is
  what prevents look-ahead.

In [ ]:
bdays = pd.bdate_range(START, pd.Timestamp.today().normalize())
def on_bdays(s): return s.reindex(bdays.union(s.index)).ffill().reindex(bdays) if len(s) else pd.Series(np.nan, bdays, name=s.name)

D = pd.concat([on_bdays(s) for s in [policy, tbill3m, y2, y10, ylong, fedf, us2, wti, usdcad]], axis=1)

# ETF prices on the same calendar
px = pd.DataFrame({k: on_bdays(v) for k, v in ca_px.items()}, index=bdays)
for zname, s in us_px.items():                          # US ETF in CAD
    px[zname] = on_bdays(s) * D["usdcad"]

# ZFL splice: proxy return before inception, real afterwards
zfl_proxy_ret = (-ZFL_DURATION * D["ylong"].diff() / 100.0)
zfl_real_ret  = px["ZFL"].pct_change() if "ZFL" in px else pd.Series(np.nan, bdays)
zfl_ret = zfl_real_ret.where(zfl_real_ret.notna() & px.get("ZFL", pd.Series(np.nan, bdays)).notna(), zfl_proxy_ret)
px["ZFL"] = (1 + zfl_ret.fillna(0)).cumprod()
print("ZFL real data starts:", ca_px["ZFL"].index[0].date() if "ZFL" in ca_px else "n/a", "| proxy covers earlier")

# Monthly macro with release lag -> daily
def release_lag(s, days):
    if s.empty: return pd.Series(np.nan, bdays, name=s.name)
    s = s.copy(); s.index = s.index + pd.offsets.MonthEnd(0) + pd.Timedelta(days=days)
    return on_bdays(s.groupby(level=0).last())

M = pd.concat([release_lag(cpi_tot, 24), release_lag(cpi_trm, 24),
               release_lag(cpi_med, 24), release_lag(unemp, 10)], axis=1)
D = D.join(M)
print("daily panel:", D.shape, "| ETF panel:", px.shape)

## 5. Label every meeting

Decision = policy rate a few days after the meeting minus the rate the day before.
Unscheduled changes (no scheduled date within 3 days) become their own meetings.

In [ ]:
pol = policy.copy()
changes = pol.index[pol.diff().abs() > 1e-9]
extra = [d for d in changes if (abs(SCHEDULED - d) <= pd.Timedelta(days=3)).sum() == 0]
meet = SCHEDULED.union(pd.DatetimeIndex(extra))
meet = meet[(meet >= FIRST_MEET)]
print("unscheduled moves added:", [d.date() for d in extra if d >= pd.Timestamp(FIRST_MEET)])

def rate_asof(d):
    s = pol[pol.index <= d]; return s.iloc[-1] if len(s) else np.nan

rows = []
for d in meet:
    before = rate_asof(d - pd.Timedelta(days=1))
    after  = rate_asof(d + pd.Timedelta(days=5)) if pol.index[-1] >= d + pd.Timedelta(days=1) else np.nan
    rows.append({"date": d, "before": before, "after": after})
L = pd.DataFrame(rows).set_index("date")
L["change"] = L["after"] - L["before"]
L["y"] = np.select([L["change"] < -LABEL_EPS, L["change"] > LABEL_EPS], [0, 2], 1)
L.loc[L["change"].isna(), "y"] = -1              # future / not yet known
past = L[L["y"] >= 0]
print(f"labelled meetings: {len(past)}   |   future meetings: {(L['y']<0).sum()}")
print(past["y"].map(dict(enumerate(CLASSES))).value_counts().to_string())
past.tail(8)

## 6. Feature engineering (as of the business day before each meeting)

| Block | Features |
|---|---|
| Market-implied path | 3m T-bill - policy, 2y - policy, 10y - 2y slope, 20/60d change in 2y |
| Global | Fed funds - policy, CA 2y - US 2y, 20d change in US 2y, WTI and USD/CAD returns |
| Inflation / labour | CPI y/y, CPI-trim, CPI-median, core vs 2% target, 3m change in core, unemployment level and 3m/6m change |
| Policy state | policy level, last decision, net direction of last 4 decisions, meetings since last change |
| Assets | ZFL 5/20/60d return and 20d vol; each sector ETF 20/60d return **relative to XIU** |

In [ ]:
ret = px.pct_change(fill_method=None)
def cumret(n): return px / px.shift(n) - 1

R20, R60, R5 = cumret(20), cumret(60), cumret(5)
F = pd.DataFrame(index=bdays)
F["tbill_spread"] = D["tbill3m"] - D["policy"]
F["y2_spread"]    = D["y2"] - D["policy"]
F["slope_10_2"]   = D["y10"] - D["y2"]
F["d_y2_20"]      = D["y2"].diff(20)
F["d_y2_60"]      = D["y2"].diff(60)
F["fed_minus_boc"]= D["fedfunds"] - D["policy"]
F["ca_us_2y"]     = D["y2"] - D["us2y"]
F["d_us2y_20"]    = D["us2y"].diff(20)
F["wti_r60"]      = D["wti"].pct_change(60, fill_method=None)
F["usdcad_r20"]   = D["usdcad"].pct_change(20, fill_method=None)
F["usdcad_r60"]   = D["usdcad"].pct_change(60, fill_method=None)
F["cpi_yoy"]      = D["cpi_yoy"]
F["cpi_trim"]     = D["cpi_trim"]
F["cpi_median"]   = D["cpi_median"]
core = D[["cpi_trim","cpi_median"]].mean(axis=1)
F["core_gap"]     = core - 2.0
F["d_core_3m"]    = core - core.shift(63)
F["unemp"]        = D["unemp"]
F["d_unemp_3m"]   = D["unemp"] - D["unemp"].shift(63)
F["d_unemp_6m"]   = D["unemp"] - D["unemp"].shift(126)
F["policy_level"] = D["policy"]

F["zfl_r5"], F["zfl_r20"], F["zfl_r60"] = R5["ZFL"], R20["ZFL"], R60["ZFL"]
F["zfl_vol20"] = ret["ZFL"].rolling(20).std() * np.sqrt(252)
sector_cols = [c for c in px.columns if c not in ("ZFL", "XIU")]
for c in sector_cols:
    F[f"{c}_rel20"] = R20[c] - R20["XIU"]
    F[f"{c}_rel60"] = R60[c] - R60["XIU"]
F = F.replace([np.inf, -np.inf], np.nan)

def asof_row(frame, d):
    sub = frame[frame.index < d]
    return sub.iloc[-1] if len(sub) else pd.Series(np.nan, frame.columns)

def policy_state(d, labels):
    hist = labels[(labels.index < d) & (labels["y"] >= 0)]["y"] - 1   # -1 cut, 0 hold, +1 hike
    last = hist.iloc[-1] if len(hist) else 0
    net4 = hist.iloc[-4:].sum() if len(hist) else 0
    nz = np.where(hist.values != 0)[0]
    since = len(hist) - 1 - nz[-1] if len(nz) else len(hist)
    return pd.Series({"last_dec": last, "net_last4": net4, "meets_since_chg": min(since, 24)})

X_all = pd.DataFrame([pd.concat([asof_row(F, d), policy_state(d, L)]) for d in L.index], index=L.index)
y_all = L["y"]
cov = X_all[y_all >= 0].notna().mean().sort_values()
dropped = cov[cov < 0.25].index.tolist()          # too little history to learn from
X_all = X_all.drop(columns=dropped); cov = cov.drop(dropped)
print("feature matrix:", X_all.shape, "| dropped for coverage <25%:", dropped)
print("lowest coverage features:\n", cov.head(8).round(2).to_string())

## 7. Common tools: safe classifier, CV splitter, metrics

Early CV folds can be missing a class (e.g. no hikes in 2013-2016). `Safe` fits on
whatever classes are present and always returns three probability columns, so every
model and fold scores on the same footing. The main metric is **log loss** (it rewards
calibrated probabilities, which is what a rate-decision model should produce); accuracy
and macro-F1 are reported alongside.

In [ ]:
class Safe(BaseEstimator, ClassifierMixin):
    def __init__(self, base=None): self.base = base
    def fit(self, X, y):
        y = np.asarray(y); self.present_ = np.unique(y)
        if len(self.present_) == 1:
            self.model_ = None
        else:
            remap = {c: i for i, c in enumerate(self.present_)}
            self.model_ = clone(self.base).fit(X, np.array([remap[v] for v in y]))
        self.classes_ = np.arange(3); return self
    def predict_proba(self, X):
        P = np.full((len(X), 3), 1e-4)
        if self.model_ is None: P[:, self.present_[0]] = 1
        else: P[:, self.present_] = self.model_.predict_proba(X)
        return P / P.sum(1, keepdims=True)
    def predict(self, X): return self.predict_proba(X).argmax(1)

def expanding_cv(n, n_splits=4, min_train=40):
    min_train = min(min_train, max(10, n // 2))      # adapt if the tuning window is short
    fold = max(1, (n - min_train) // n_splits)
    for k in range(n_splits):
        tr_end = min_train + k * fold
        te_end = n if k == n_splits - 1 else tr_end + fold
        if tr_end < n: yield np.arange(tr_end), np.arange(tr_end, te_end)

def ll_scorer(est, Xv, yv): return -log_loss(yv, est.predict_proba(Xv), labels=[0, 1, 2])

def report(name, P, y):
    return {"model": name, "log_loss": log_loss(y, P, labels=[0,1,2]),
            "accuracy": accuracy_score(y, P.argmax(1)),
            "macro_f1": f1_score(y, P.argmax(1), average="macro", labels=[0,1,2], zero_division=0)}

lab = y_all[y_all >= 0].astype(int)
X   = X_all.loc[lab.index]
tune_mask = lab.index < VALID_START
wf_idx    = np.where(lab.index >= VALID_START)[0]
assert tune_mask.sum() >= 30, (f"only {tune_mask.sum()} labelled meetings before {VALID_START}: "
    "check the download printout in section 3 (policy rate / yields coverage)")
print(f"tuning set: {tune_mask.sum()} meetings | walk-forward: {len(wf_idx)} meetings "
      f"({(lab.index[wf_idx] < TEST_START).sum()} validation, {(lab.index[wf_idx] >= TEST_START).sum()} holdout)")

## 8. Hyperparameter search (pre-2012 data only)

Randomized search with an expanding-window CV, scored on log loss. The walk-forward
period is never seen during tuning.

- **Market-only LR**: just the T-bill and 2y spreads. The honest benchmark.
- **Logistic regression**: all features, tuned L1/L2/elastic-net strength.
- **XGBoost**: depth, trees, learning rate, subsampling, L1/L2/gamma, min child weight.
- **MLP**: architecture, activation, L2 alpha, learning rate, batch size.

In [ ]:
MARKET_COLS = ["tbill_spread", "y2_spread"]
Xt, yt = X[tune_mask], lab[tune_mask]
cv = list(expanding_cv(len(Xt), n_splits=4, min_train=40))

def lin(C=1.0, penalty="l2", l1_ratio=None):
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         LogisticRegression(C=C, penalty=penalty, l1_ratio=l1_ratio,
                                            solver="saga", max_iter=5000))

SEARCH = {
  "MarketLR": (Safe(lin()), {"base__logisticregression__C": [0.03, 0.1, 0.3, 1, 3, 10]}, MARKET_COLS, 6),
  "LogReg":   (Safe(lin(penalty="elasticnet", l1_ratio=0.5)),
               {"base__logisticregression__C": [0.01, 0.03, 0.1, 0.3, 1],
                "base__logisticregression__l1_ratio": [0.0, 0.25, 0.5, 0.75, 1.0]}, None, 20),
  "XGBoost":  (Safe(XGBClassifier(objective="multi:softprob", eval_metric="mlogloss",
                                  n_jobs=-1, random_state=42, verbosity=0)),
               {"base__n_estimators": [100, 200, 400], "base__max_depth": [1, 2, 3],
                "base__learning_rate": [0.01, 0.03, 0.1], "base__subsample": [0.6, 0.8, 1.0],
                "base__colsample_bytree": [0.3, 0.5, 0.8], "base__min_child_weight": [1, 3, 5],
                "base__gamma": [0, 0.5, 1], "base__reg_alpha": [0, 0.1, 1],
                "base__reg_lambda": [1, 5, 10]}, None, 30),
  "MLP":      (Safe(make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                  MLPClassifier(max_iter=2000, early_stopping=False, random_state=42))),
               {"base__mlpclassifier__hidden_layer_sizes": [(16,), (32,), (32, 16), (64, 32)],
                "base__mlpclassifier__activation": ["relu", "tanh"],
                "base__mlpclassifier__alpha": [1e-3, 1e-2, 0.1, 1.0],
                "base__mlpclassifier__learning_rate_init": [1e-3, 3e-3]}, None, 15),
}

BEST = {}
for name, (est, space, cols, n_iter) in SEARCH.items():
    Xs = Xt[cols] if cols else Xt
    s = RandomizedSearchCV(est, space, n_iter=n_iter, scoring=ll_scorer, cv=cv,
                           random_state=42, n_jobs=-1, error_score=-9.0).fit(Xs, yt)
    BEST[name] = {"params": s.best_params_, "cols": cols, "cv_logloss": -s.best_score_}
    print(f"{name:9s} CV log loss {-s.best_score_:.3f}  params={ {k.split('__')[-1]: v for k, v in s.best_params_.items()} }")

## 9. Walk-forward test

Before each meeting from 2012 on: refit every model (with its tuned params) on all
meetings before it, predict that meeting. Baselines run the same way.

In [ ]:
def build(name):
    est, _, cols, _ = SEARCH[name]
    return clone(est).set_params(**BEST[name]["params"]), cols

WF = {n: np.zeros((len(wf_idx), 3)) for n in list(SEARCH) + ["AlwaysHold", "Persistence"]}
for j, i in enumerate(wf_idx):
    Xtr, ytr, Xte = X.iloc[:i], lab.iloc[:i], X.iloc[[i]]
    for name in SEARCH:
        m, cols = build(name)
        m.fit(Xtr[cols] if cols else Xtr, ytr)
        WF[name][j] = m.predict_proba(Xte[cols] if cols else Xte)[0]
    # baselines: unconditional hold; repeat last decision (smoothed with train frequencies)
    freq = np.bincount(ytr, minlength=3) / len(ytr)
    WF["AlwaysHold"][j] = freq
    last = int(Xte["last_dec"].iloc[0] + 1); p = 0.3 * freq; p[last] += 0.7
    WF["Persistence"][j] = p
    if j % 20 == 0: print(f"  ...{j}/{len(wf_idx)} meetings ({lab.index[i].date()})")
print("walk-forward done")

### 9b. LSTM on daily sequences (optional)

An LSTM reads the last 60 business days of the daily market/macro panel before each
meeting. With ~200 decisions it is data-hungry, so it is refit once a year in the
walk-forward instead of every meeting, and it gets a small grid. Skipped automatically if
TensorFlow is missing.

In [ ]:
RUN_LSTM, SEQ = True, 60
if RUN_LSTM:
    try:
        import tensorflow as tf
        from tensorflow.keras import layers, models
        daily = F.drop(columns=[c for c in F.columns if F[c].notna().mean() < 0.6]).ffill()

        def seq_at(d, mu, sd):
            w = daily[daily.index < d].iloc[-SEQ:]
            return ((w - mu) / sd).fillna(0).values

        def lstm_fit(idx, g):
            dates = lab.index[idx]
            ref = daily[daily.index < dates[-1]]
            mu, sd = ref.mean(), ref.std().replace(0, 1)
            Xs = np.stack([seq_at(d, mu, sd) for d in dates]); ys = lab.iloc[idx].values
            tf.random.set_seed(42)
            net = models.Sequential([layers.Input((SEQ, daily.shape[1])),
                                     layers.LSTM(g["u"], dropout=g["drop"]),
                                     layers.Dense(16, activation="relu"),
                                     layers.Dense(3, activation="softmax")])
            net.compile(tf.keras.optimizers.Adam(g["lr"]), "sparse_categorical_crossentropy")
            w = len(ys) / (3 * np.maximum(np.bincount(ys, minlength=3), 1))
            net.fit(Xs, ys, epochs=g["ep"], batch_size=16, verbose=0,
                    sample_weight=w[ys] ** 0.5)
            return net, mu, sd

        grid = [{"u": 16, "drop": 0.2, "lr": 1e-3, "ep": 60},
                {"u": 32, "drop": 0.3, "lr": 1e-3, "ep": 60},
                {"u": 16, "drop": 0.3, "lr": 3e-3, "ep": 40}]
        tune_idx = np.where(tune_mask)[0]
        scores = []
        for g in grid:                                  # tune on pre-2012, last 25% held out
            cut = int(len(tune_idx) * 0.75)
            net, mu, sd = lstm_fit(tune_idx[:cut], g)
            P = net.predict(np.stack([seq_at(d, mu, sd) for d in lab.index[tune_idx[cut:]]]), verbose=0)
            scores.append(log_loss(lab.iloc[tune_idx[cut:]], P, labels=[0, 1, 2]))
            print(f"  LSTM {g} -> tune log loss {scores[-1]:.3f}")
        g_best = grid[int(np.argmin(scores))]
        BEST["LSTM"] = {"params": g_best, "cols": None, "cv_logloss": min(scores)}

        WF["LSTM"] = np.zeros((len(wf_idx), 3)); cur_year, net = None, None
        for j, i in enumerate(wf_idx):
            d = lab.index[i]
            if d.year != cur_year:
                net, mu, sd = lstm_fit(np.arange(i), g_best); cur_year = d.year
            WF["LSTM"][j] = net.predict(seq_at(d, mu, sd)[None], verbose=0)[0]
        print("LSTM walk-forward done")
    except Exception as e:
        print(f"LSTM skipped ({type(e).__name__}: {e})")

In [ ]:
# Ensemble = mean of the tuned learners (not the baselines)
members = [k for k in ["LogReg", "XGBoost", "MLP", "LSTM"] if k in WF]
WF["Ensemble"] = np.mean([WF[k] for k in members], axis=0)
print("ensemble members:", members)

## 10. Evaluation: validation (select) vs holdout (report)

The winner is chosen on 2012-2018 only. The 2019+ numbers are the honest out-of-sample
estimate. Lower log loss is better; the always-hold line is what you get with no model.

In [ ]:
y_wf = lab.iloc[wf_idx].values
is_val = lab.index[wf_idx] < TEST_START
val = pd.DataFrame([report(k, P[is_val], y_wf[is_val]) for k, P in WF.items()]).set_index("model").sort_values("log_loss")
hol = pd.DataFrame([report(k, P[~is_val], y_wf[~is_val]) for k, P in WF.items()]).set_index("model").loc[val.index]

candidates = [k for k in val.index if k not in ("AlwaysHold", "Persistence")]
best_name = val.loc[candidates, "log_loss"].idxmin()

print("VALIDATION 2012-2018 (used to pick the model)\n", val.round(3).to_string(), "\n")
print("HOLDOUT 2019+ (reported)\n", hol.round(3).to_string(), "\n")
print(f"Selected on validation: {best_name}")

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(val)); w = 0.38
ax.bar(x - w/2, val["log_loss"], w, label="validation 2012-18", color="#9DB4D3")
ax.bar(x + w/2, hol["log_loss"], w, label="holdout 2019+", color="#4C72B0")
ax.axhline(hol.loc["AlwaysHold", "log_loss"], color="red", ls="--", lw=1, label="always-hold (holdout)")
ax.set_xticks(x); ax.set_xticklabels(val.index, rotation=20); ax.set_ylabel("log loss (lower = better)")
ax.legend(); ax.set_title("Walk-forward log loss"); plt.tight_layout(); plt.show()

cm = confusion_matrix(y_wf[~is_val], WF[best_name][~is_val].argmax(1), labels=[0, 1, 2])
print(f"Holdout confusion matrix ({best_name}), rows = actual, cols = predicted")
print(pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_string())

In [ ]:
# Probability path vs what the BoC did (holdout)
d_ho = lab.index[wf_idx][~is_val]; P = WF[best_name][~is_val]
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.stackplot(d_ho, P[:, 0], P[:, 1], P[:, 2], labels=["P(cut)", "P(hold)", "P(hike)"],
             colors=["#C44E52", "#CCCCCC", "#55A868"], alpha=0.85, step="post")
act = y_wf[~is_val]
ax.scatter(d_ho[act == 0], np.full((act == 0).sum(), 1.03), marker="v", color="#C44E52", label="actual cut", clip_on=False)
ax.scatter(d_ho[act == 2], np.full((act == 2).sum(), 1.03), marker="^", color="#55A868", label="actual hike", clip_on=False)
ax.set_ylim(0, 1); ax.set_title(f"{best_name}: predicted probabilities before each meeting (holdout)")
ax.legend(ncol=5, fontsize=8, loc="lower left"); plt.tight_layout(); plt.show()

## 11. What drives it

XGBoost gain importance and the logistic-regression coefficients (standardized, so
comparable). Read them as associations, not causal effects.

In [ ]:
xgb_full, _ = build("XGBoost"); xgb_full.fit(X, lab)
imp = pd.Series(xgb_full.model_.feature_importances_, index=X.columns).sort_values(ascending=False).head(15)
lr_full, _ = build("LogReg"); lr_full.fit(X, lab)
coef = lr_full.model_[-1].coef_
lr_imp = pd.Series(np.abs(coef).max(0), index=X.columns).sort_values(ascending=False).head(15)

fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
axs[0].barh(imp.index[::-1], imp.values[::-1], color="#4C72B0"); axs[0].set_title("XGBoost importance (top 15)")
axs[1].barh(lr_imp.index[::-1], lr_imp.values[::-1], color="#8172B3"); axs[1].set_title("LogReg |coef| max over classes")
plt.tight_layout(); plt.show()

## 12. Store the best configuration

Same idea as `MODEL_STORE` in the Lotto Max notebook: every model's tuned parameters next
to its scores, the winner flagged, and a JSON dump you can reload to skip the search.

In [ ]:
MODEL_STORE = {k: {"params": BEST.get(k, {}).get("params", "mean of members" if k == "Ensemble" else "baseline"),
                   "val_logloss": float(val.loc[k, "log_loss"]),
                   "holdout_logloss": float(hol.loc[k, "log_loss"]),
                   "holdout_acc": float(hol.loc[k, "accuracy"])} for k in WF}
with open("best_params_boc.json", "w") as f:
    json.dump({"best_model": best_name, "store": MODEL_STORE}, f, indent=2, default=str)
for k, v in sorted(MODEL_STORE.items(), key=lambda kv: kv[1]["val_logloss"]):
    flag = "  <- SELECTED" if k == best_name else ""
    print(f"  {k:12s} val {v['val_logloss']:.3f} | holdout {v['holdout_logloss']:.3f} / acc {v['holdout_acc']:.2f}{flag}")
print("Saved best_params_boc.json")

## 13. Prediction for the next meeting

Refits the selected approach on every labelled meeting and scores the feature row as of
the latest available data. The market-only model is printed next to it: if the two
disagree a lot, that is the interesting part.

In [ ]:
future = L[(L["y"] < 0) & (L.index >= pd.Timestamp.today().normalize())]
next_d = future.index[0] if len(future) else SCHEDULED[SCHEDULED > pd.Timestamp.today()][0]
x_next = pd.DataFrame([pd.concat([asof_row(F, next_d), policy_state(next_d, L)])], index=[next_d])[X.columns]

def fit_predict(name):
    if name == "LSTM":
        net, mu, sd = lstm_fit(np.arange(len(lab)), BEST["LSTM"]["params"])
        return net.predict(seq_at(next_d, mu, sd)[None], verbose=0)[0]
    m, cols = build(name); m.fit(X[cols] if cols else X, lab)
    return m.predict_proba(x_next[cols] if cols else x_next)[0]

full_models = [k for k in val.index if k not in ("AlwaysHold", "Persistence", "MarketLR")]
best_full = full_models[0]                        # best feature-rich approach on validation
show = list(dict.fromkeys([best_name, best_full, "MarketLR"]))
probs = {}
for n in show:
    parts = members if n == "Ensemble" else [n]
    probs[n] = np.mean([fit_predict(m) for m in parts], axis=0)
cur = policy.iloc[-1]

print("=" * 60)
print(f"  NEXT BoC DECISION: {next_d.date()}   (policy rate now {cur:.2f}%)")
print("=" * 60)
for lbl, p in probs.items():
    tag = " (selected)" if lbl == best_name else ""
    print(f"  {lbl+tag:20s} cut {p[0]:5.1%} | hold {p[1]:5.1%} | hike {p[2]:5.1%}  -> {CLASSES[p.argmax()].upper()}")
print(f"  inputs as of {F.dropna(how='all').index[-1].date()}: T-bill spread {x_next['tbill_spread'].iloc[0]:+.2f} pp, "
      f"2y spread {x_next['y2_spread'].iloc[0]:+.2f} pp, core CPI gap {x_next['core_gap'].iloc[0]:+.2f} pp")
print(f"  selected model holdout log loss {MODEL_STORE[best_name]['holdout_logloss']:.3f} "
      f"(always-hold {MODEL_STORE['AlwaysHold']['holdout_logloss']:.3f})")
print("=" * 60)

## 14. Test any past meeting (out-of-sample)

Set `MEETING` to any decision date in the calendar. Every model is trained only on
meetings **before** that date, then predicts it, so the result is what the notebook would
have said going into that meeting. The actual outcome prints underneath.

In [ ]:
MEETING = "2026-10-28"                                # change to any meeting date

d = pd.Timestamp(MEETING)
assert d in X_all.index, f"{d.date()} is not in the meeting calendar"

x_d   = X_all.loc[[d], X.columns]
prior = lab.index < d
rows  = {}
for name in SEARCH:                                   # MarketLR, LogReg, XGBoost, MLP
    m, cols = build(name)
    m.fit(X.loc[prior, cols] if cols else X.loc[prior], lab[prior])
    rows[name] = m.predict_proba(x_d[cols] if cols else x_d)[0]
if "LSTM" in BEST:
    net, mu, sd = lstm_fit(np.where(prior)[0], BEST["LSTM"]["params"])
    rows["LSTM"] = net.predict(seq_at(d, mu, sd)[None], verbose=0)[0]
rows["Ensemble"] = np.mean([rows[k] for k in members if k in rows], axis=0)

out = pd.DataFrame(rows, index=CLASSES).T
out["call"] = out[CLASSES].idxmax(axis=1).str.upper()
out.loc[best_name, "call"] += "  <- selected"
for c in CLASSES: out[c] = out[c].map("{:.1%}".format)

print(f"Meeting {d.date()}  |  rate before {L.loc[d, 'before']:.2f}%  |  trained on {prior.sum()} earlier meetings")
print(out.to_string())
actual = L.loc[d, "y"]
if actual >= 0:
    print(f"Actual: {CLASSES[int(actual)].upper()} (change {L.loc[d, 'change']:+.2f} pp)")
else:
    print("Actual: not in the data yet")